# XGBoost One-vs-Rest vs. CatBoost

A public discussion thread on this competition (Masaya Kawamata, ["CV vs Public LB across 13 model families"](https://www.kaggle.com/competitions/playground-series-s6e7/discussion/718258))
found that `XGB_OvR` — XGBoost trained as three independent one-vs-rest binary
classifiers and combined via argmax — scored highest of 13 model families tested
(CV 0.95036 / LB 0.95040), modestly ahead of native-multiclass XGBoost variants.
That is the formulation we test here against our own CatBoost baseline.

Earlier rounds of threshold tuning and ensembling on this dataset have repeatedly
run into the same synthesis-noise ceiling, so the expected upside from a new
model family is small (roughly 0.0003-0.0005, in line with the noise band
reported in that same discussion thread) — but it is worth trying since it is a
genuinely untried formulation rather than a rehash of blend weights or
thresholds.

We compare three models: an XGBoost one-vs-rest ensemble on the engineered
feature set, a CatBoost model reproducing our earlier best configuration on the
base feature set (CatBoost-V1), and a second CatBoost model on the same
engineered feature set as the XGBoost ensemble (CatBoost-V2). Using the same
engineered features for both XGB-OvR and CatBoost-V2 isolates the algorithm
difference (XGBoost one-vs-rest vs. CatBoost native multiclass) from the
feature-set difference, which a single CatBoost baseline alone would confound.


In [1]:
import warnings
# CatBoost tries to numba-JIT our custom eval metric and fails (harmless,
# just noisy); silence the warning.
warnings.filterwarnings('ignore', category=UserWarning, module='catboost')

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from tqdm.auto import tqdm
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

print('xgboost version:', xgb.__version__)

import os, glob
DATA_DIR = Path('/kaggle/input/playground-series-s6e7')
if not (DATA_DIR / 'train.csv').exists():
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if hits:
        DATA_DIR = Path(hits[0]).parent
    else:
        DATA_DIR = Path('..') / 'data'  # local fallback for testing outside Kaggle

TARGET = 'health_condition'
NUMERIC_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
                 'step_count', 'exercise_duration', 'water_intake']
CATEGORICAL_COLS = ['diet_type', 'stress_level', 'sleep_quality',
                     'physical_activity_level', 'smoking_alcohol', 'gender']
BASE_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

for col in CATEGORICAL_COLS:
    train[col] = train[col].fillna('missing').astype(str)
    test[col] = test[col].fillna('missing').astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train[TARGET])
n_classes = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))
print('train', train.shape, 'test', test.shape)

sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

xgboost version: 3.3.0
classes: ['at-risk', 'fit', 'unhealthy']
train (690088, 15) test (295753, 14)


## Reconstruct the 35-feature engineered set (same as v0.4/v0.5)

Shared by XGB-OvR and CatBoost-V2 (both use the engineered feature set).

In [2]:
train['sleep_duration_bin'], sleep_bin_edges = pd.qcut(
    train['sleep_duration'], q=10, duplicates='drop', retbins=True)
test_sleep_clipped = test['sleep_duration'].clip(lower=sleep_bin_edges[0], upper=sleep_bin_edges[-1])
test['sleep_duration_bin'] = pd.cut(test_sleep_clipped, bins=sleep_bin_edges, include_lowest=True)

train['sleep_duration_bin'] = train['sleep_duration_bin'].astype(str)
train.loc[train['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'
test['sleep_duration_bin'] = test['sleep_duration_bin'].astype(str)
test.loc[test['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'

MISSING_INDICATOR_COLS = []
for col in NUMERIC_COLS:
    ind_col = f'is_missing_{col}'
    train[ind_col] = train[col].isnull().astype(int)
    test[ind_col] = test[col].isnull().astype(int)
    MISSING_INDICATOR_COLS.append(ind_col)

def make_cross(df, col_a, col_b, name):
    df[name] = df[col_a].astype(str) + '_' + df[col_b].astype(str)

make_cross(train, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(test, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(train, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(test, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(train, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')
make_cross(test, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')

CROSS_COLS = ['stress_x_activity', 'sleepq_x_smoking', 'sleepbin_x_stress']

def oof_target_encode(train_col, test_col, y, n_classes, folds, alpha=20):
    n_train = len(train_col)
    oof_encoded = np.zeros((n_train, n_classes))
    test_encoded = np.zeros((len(test_col), n_classes))
    class_cols = [f'k{i}' for i in range(n_classes)]
    onehot = pd.get_dummies(pd.Series(y)).values
    for tr_idx, val_idx in folds:
        cats_tr = pd.Series(train_col.iloc[tr_idx].astype(str).values)
        y_tr_onehot = onehot[tr_idx]
        prior = pd.Series(y_tr_onehot.mean(axis=0), index=class_cols)
        stats = pd.DataFrame(y_tr_onehot, columns=class_cols)
        stats['cat'] = cats_tr.values
        grp_sum = stats.groupby('cat')[class_cols].sum()
        grp_count = stats.groupby('cat').size()
        enc_map = grp_sum.add(alpha * prior, axis=1).div(grp_count + alpha, axis=0)
        val_cats = pd.Series(train_col.iloc[val_idx].astype(str).values)
        oof_encoded[val_idx] = enc_map.reindex(val_cats).fillna(prior).values
        test_cats = pd.Series(test_col.astype(str).values)
        test_encoded += enc_map.reindex(test_cats).fillna(prior).values / len(folds)
    return oof_encoded, test_encoded

TARGET_ENCODE_COLS = ['stress_level', 'physical_activity_level', 'stress_x_activity', 'sleepq_x_smoking']
TARGET_ENCODED_FEATURE_COLS = []
for col in tqdm(TARGET_ENCODE_COLS, desc='target encoding'):
    oof_enc, test_enc = oof_target_encode(train[col], test[col], y, n_classes, folds)
    for k in range(n_classes):
        fcol = f'te_{col}_k{k}'
        train[fcol] = oof_enc[:, k]
        test[fcol] = test_enc[:, k]
        TARGET_ENCODED_FEATURE_COLS.append(fcol)

ENGINEERED_CATEGORICAL_COLS = CATEGORICAL_COLS + CROSS_COLS
ENGINEERED_FEATURES = (NUMERIC_COLS + CATEGORICAL_COLS + MISSING_INDICATOR_COLS
                        + CROSS_COLS + TARGET_ENCODED_FEATURE_COLS)
print(f'{len(ENGINEERED_FEATURES)} engineered features (matches v0.2/v0.3 Variant 2)')

target encoding:   0%|          | 0/4 [00:00<?, ?it/s]

35 engineered features (matches v0.2/v0.3 Variant 2)


## XGBoost native categorical dtype setup

Like LightGBM, XGBoost's `enable_categorical=True` needs `pandas.Categorical`
dtype with a shared category list between train and test.

In [3]:
train_xgb = train.copy()
test_xgb = test.copy()
for col in ENGINEERED_CATEGORICAL_COLS:
    categories = sorted(set(train_xgb[col].unique()) | set(test_xgb[col].unique()))
    train_xgb[col] = pd.Categorical(train_xgb[col], categories=categories)
    test_xgb[col] = pd.Categorical(test_xgb[col], categories=categories)

## Member A — XGBoost one-vs-rest

3 independent binary classifiers per fold (one per class), `binary:logistic`,
`scale_pos_weight` for imbalance (XGBoost's analogue to `class_weight='balanced'`).
Combined via argmax over the 3 raw probabilities — the standard OvR combination.

**Structural note**: each binary classifier early-stops on its own binary AUC —
there's no single early-stopping signal that corresponds to the final combined
multiclass balanced accuracy in a true OvR setup, unlike our native-multiclass
CatBoost/LightGBM models which early-stop directly on balanced accuracy.

In [4]:
oof_proba_ovr = np.zeros((len(train_xgb), n_classes))
test_proba_ovr = np.zeros((len(test_xgb), n_classes))
binary_fold_aucs = {c: [] for c in range(n_classes)}

for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc='XGB-OvR folds')):
    X_tr = train_xgb[ENGINEERED_FEATURES].iloc[tr_idx]
    X_val = train_xgb[ENGINEERED_FEATURES].iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    for c in tqdm(range(n_classes), desc=f'fold {fold} classes', leave=False):
        y_tr_bin = (y_tr == c).astype(int)
        y_val_bin = (y_val == c).astype(int)
        pos = y_tr_bin.sum()
        neg = len(y_tr_bin) - pos
        scale_pos_weight = neg / pos
        model = xgb.XGBClassifier(
            objective='binary:logistic', eval_metric='auc',
            scale_pos_weight=scale_pos_weight,
            n_estimators=2000, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            enable_categorical=True, random_state=42, n_jobs=-1,
            early_stopping_rounds=50,
        )
        model.fit(X_tr, y_tr_bin, eval_set=[(X_val, y_val_bin)], verbose=False)
        oof_proba_ovr[val_idx, c] = model.predict_proba(X_val)[:, 1]
        test_proba_ovr[:, c] += model.predict_proba(test_xgb[ENGINEERED_FEATURES])[:, 1] / N_FOLDS
        val_auc = model.evals_result()['validation_0']['auc'][model.best_iteration]
        binary_fold_aucs[c].append(val_auc)

oof_pred_ovr = oof_proba_ovr.argmax(axis=1)
oof_bal_acc_ovr = balanced_accuracy_score(y, oof_pred_ovr)
print('per-class binary AUC by fold:')
for c in range(n_classes):
    print(f'  {label_encoder.classes_[c]}: {[round(a, 4) for a in binary_fold_aucs[c]]}')
print(f'\nXGB-OvR OOF balanced accuracy: {oof_bal_acc_ovr:.4f}')
print()
print(classification_report(y, oof_pred_ovr, target_names=label_encoder.classes_, digits=4))

XGB-OvR folds:   0%|          | 0/5 [00:00<?, ?it/s]

fold 0 classes:   0%|          | 0/3 [00:00<?, ?it/s]

fold 1 classes:   0%|          | 0/3 [00:00<?, ?it/s]

fold 2 classes:   0%|          | 0/3 [00:00<?, ?it/s]

fold 3 classes:   0%|          | 0/3 [00:00<?, ?it/s]

fold 4 classes:   0%|          | 0/3 [00:00<?, ?it/s]

per-class binary AUC by fold:
  at-risk: [0.9813, 0.9807, 0.9802, 0.9817, 0.9796]
  fit: [0.9832, 0.9835, 0.9822, 0.9823, 0.9823]
  unhealthy: [0.9879, 0.9878, 0.9875, 0.9885, 0.9875]

XGB-OvR OOF balanced accuracy: 0.9493

              precision    recall  f1-score   support

     at-risk     0.9932    0.9381    0.9649    592561
         fit     0.7384    0.9482    0.8302     39803
   unhealthy     0.7005    0.9617    0.8106     57724

    accuracy                         0.9407    690088
   macro avg     0.8107    0.9493    0.8686    690088
weighted avg     0.9540    0.9407    0.9442    690088



## CatBoost shared setup (custom eval metric + progress callback)

Used by both CatBoost members (Variant 1 and Variant 2).

In [5]:
class BalancedAccuracyMetric:
    def is_max_optimal(self):
        return True
    def evaluate(self, approxes, target, weight):
        approx = np.array(approxes)
        pred = approx.argmax(axis=0)
        y_true = np.array(target).astype(int)
        return balanced_accuracy_score(y_true, pred), 1.0
    def get_final_error(self, error, weight):
        return error

class TqdmCatBoostCallback:
    def __init__(self, total, desc):
        self.pbar = tqdm(total=total, desc=desc, leave=False)
        self.last = 0
    def after_iteration(self, info):
        self.pbar.update(info.iteration - self.last)
        self.last = info.iteration
        return True

def run_catboost_cv_with_proba(feature_frame, test_frame, categorical_features, folds, y, n_classes, desc):
    oof_proba = np.zeros((len(feature_frame), n_classes))
    test_proba = np.zeros((len(test_frame), n_classes))
    fold_scores, best_iters = [], []
    for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc=desc)):
        X_tr, X_val = feature_frame.iloc[tr_idx], feature_frame.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = CatBoostClassifier(
            loss_function='MultiClass', eval_metric=BalancedAccuracyMetric(),
            auto_class_weights='Balanced', iterations=3000, learning_rate=0.05, depth=6,
            l2_leaf_reg=3, random_seed=42, task_type='CPU', verbose=False,
        )
        round_pbar = TqdmCatBoostCallback(3000, f'{desc} fold {fold} rounds')
        model.fit(X_tr, y_tr, cat_features=categorical_features, eval_set=(X_val, y_val),
                  early_stopping_rounds=100, callbacks=[round_pbar])
        round_pbar.pbar.close()
        oof_proba[val_idx] = model.predict_proba(X_val)
        val_pred = oof_proba[val_idx].argmax(axis=1)
        fold_scores.append(balanced_accuracy_score(y_val, val_pred))
        best_iters.append(model.best_iteration_)
        test_proba += model.predict_proba(test_frame) / len(folds)
    oof_pred = oof_proba.argmax(axis=1)
    oof_bal_acc = balanced_accuracy_score(y, oof_pred)
    return {'oof_proba': oof_proba, 'test_proba': test_proba, 'fold_scores': fold_scores,
            'best_iters': best_iters, 'oof_balanced_accuracy': oof_bal_acc}

## Member B — CatBoost-V1 (reproducing v0.3's exact config, base features)

In [6]:
result_cb_v1 = run_catboost_cv_with_proba(train[BASE_FEATURES], test[BASE_FEATURES], CATEGORICAL_COLS,
                                           folds, y, n_classes, desc='CatBoost-V1')
print('best_iterations:', result_cb_v1['best_iters'])
print(f"CatBoost-V1 OOF balanced accuracy: {result_cb_v1['oof_balanced_accuracy']:.4f}")
print('v0.3 Variant 1 known result:       0.9493, best_iterations [428, 950, 605, 339, 779]')

CatBoost-V1:   0%|          | 0/5 [00:00<?, ?it/s]

CatBoost-V1 fold 0 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

CatBoost-V1 fold 1 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

CatBoost-V1 fold 2 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

CatBoost-V1 fold 3 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

CatBoost-V1 fold 4 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

best_iterations: [428, 950, 605, 339, 779]
CatBoost-V1 OOF balanced accuracy: 0.9493
v0.3 Variant 1 known result:       0.9493, best_iterations [428, 950, 605, 339, 779]


In [7]:
assert abs(result_cb_v1['oof_balanced_accuracy'] - 0.9493) < 0.002, 'CatBoost-V1 reproduction did not match v0.3 Variant 1 closely enough'
print('CatBoost-V1 reproduction check: PASS')

CatBoost-V1 reproduction check: PASS


## Member C — CatBoost-V2 (reproducing v0.3's exact config, engineered features)

Same engineered feature set as XGB-OvR (Member A) — the cleaner comparison peg,
isolating the algorithm difference from the feature-set difference that
CatBoost-V1 alone would confound.


In [8]:
result_cb_v2 = run_catboost_cv_with_proba(train[ENGINEERED_FEATURES], test[ENGINEERED_FEATURES],
                                           ENGINEERED_CATEGORICAL_COLS, folds, y, n_classes, desc='CatBoost-V2')
print('best_iterations:', result_cb_v2['best_iters'])
print(f"CatBoost-V2 OOF balanced accuracy: {result_cb_v2['oof_balanced_accuracy']:.4f}")
print('v0.3 Variant 2 known result:       0.9491, best_iterations [544, 765, 542, 354, 628]')

CatBoost-V2:   0%|          | 0/5 [00:00<?, ?it/s]

CatBoost-V2 fold 0 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

CatBoost-V2 fold 1 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

CatBoost-V2 fold 2 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

CatBoost-V2 fold 3 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

CatBoost-V2 fold 4 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

best_iterations: [544, 765, 542, 354, 628]
CatBoost-V2 OOF balanced accuracy: 0.9491
v0.3 Variant 2 known result:       0.9491, best_iterations [544, 765, 542, 354, 628]


In [9]:
assert abs(result_cb_v2['oof_balanced_accuracy'] - 0.9491) < 0.002, 'CatBoost-V2 reproduction did not match v0.3 Variant 2 closely enough'
print('CatBoost-V2 reproduction check: PASS')

CatBoost-V2 reproduction check: PASS


## Solo comparison

In [10]:
OOF_PROBAS = {
    'xgb_ovr': oof_proba_ovr,
    'catboost_v1': result_cb_v1['oof_proba'],
    'catboost_v2': result_cb_v2['oof_proba'],
}
TEST_PROBAS = {
    'xgb_ovr': test_proba_ovr,
    'catboost_v1': result_cb_v1['test_proba'],
    'catboost_v2': result_cb_v2['test_proba'],
}
MEMBER_NAMES = ['xgb_ovr', 'catboost_v1', 'catboost_v2']

solo_scores = {
    'xgb_ovr': oof_bal_acc_ovr,
    'catboost_v1': result_cb_v1['oof_balanced_accuracy'],
    'catboost_v2': result_cb_v2['oof_balanced_accuracy'],
}
for name, score in sorted(solo_scores.items(), key=lambda kv: -kv[1]):
    print(f'{score:.4f}  {name}')

0.9493  xgb_ovr
0.9493  catboost_v1
0.9491  catboost_v2


## Blend weight search (3-way simplex grid) + nested validation

Blend = weighted average of the 3 OOF probability matrices, argmax. Nested
validation (fit weights on 4/5 folds, evaluate on the held-out 5th) gives the
honest improvement estimate, the same discipline used for our earlier threshold
and ensembling experiments.


In [11]:
def simplex_grid(n_members, step=0.05):
    n_steps = round(1 / step)
    def rec(remaining_steps, remaining_members):
        if remaining_members == 1:
            yield (remaining_steps * step,)
            return
        for i in range(remaining_steps + 1):
            for rest in rec(remaining_steps - i, remaining_members - 1):
                yield (i * step,) + rest
    return list(rec(n_steps, n_members))

def blend_predict(probas_dict, weights, members):
    blended = sum(w * probas_dict[m] for w, m in zip(weights, members))
    return blended.argmax(axis=1)

def grid_search_blend(oof_probas_dict, y_true, members, grid):
    best_score, best_w = -1, tuple(1 / len(members) for _ in members)
    for w in tqdm(grid, desc=f'blend grid ({"+".join(members)})', leave=False):
        pred = blend_predict(oof_probas_dict, w, members)
        score = balanced_accuracy_score(y_true, pred)
        if score > best_score:
            best_score, best_w = score, w
    return best_score, best_w

grid3 = simplex_grid(len(MEMBER_NAMES), step=0.05)
print(f'3-way grid size: {len(grid3)} combinations')
best_score_3way, best_w_3way = grid_search_blend(OOF_PROBAS, y, MEMBER_NAMES, grid3)
print(f'3-way blend full-OOF grid search: best {best_score_3way:.4f} at weights {dict(zip(MEMBER_NAMES, (round(x,2) for x in best_w_3way)))}')
print(f'Best solo model:                  {max(solo_scores.values()):.4f}')
print(f'Improvement (same-data, likely optimistic): {best_score_3way - max(solo_scores.values()):+.4f}')

3-way grid size: 231 combinations


blend grid (xgb_ovr+catboost_v1+catboost_v2):   0%|          | 0/231 [00:00<?, ?it/s]

3-way blend full-OOF grid search: best 0.9495 at weights {'xgb_ovr': 0.55, 'catboost_v1': 0.2, 'catboost_v2': 0.25}
Best solo model:                  0.9493
Improvement (same-data, likely optimistic): +0.0002


In [12]:
nested_scores_solo_best = []
nested_scores_blend = []
nested_weights_3way = []
best_solo_key = max(solo_scores, key=solo_scores.get)

for fold_idx, (_, val_idx) in enumerate(tqdm(folds, desc='nested CV (3-way blend)')):
    other_idx = np.concatenate([folds[j][1] for j in range(N_FOLDS) if j != fold_idx])
    other_oof = {m: OOF_PROBAS[m][other_idx] for m in MEMBER_NAMES}
    _, w_fold = grid_search_blend(other_oof, y[other_idx], MEMBER_NAMES, grid3)

    val_oof = {m: OOF_PROBAS[m][val_idx] for m in MEMBER_NAMES}
    blend_pred = blend_predict(val_oof, w_fold, MEMBER_NAMES)
    solo_pred = OOF_PROBAS[best_solo_key][val_idx].argmax(axis=1)

    nested_scores_solo_best.append(balanced_accuracy_score(y[val_idx], solo_pred))
    nested_scores_blend.append(balanced_accuracy_score(y[val_idx], blend_pred))
    nested_weights_3way.append(w_fold)

print('Per-fold blend weights (fit on the other 4 folds):', [tuple(round(x, 2) for x in w) for w in nested_weights_3way])
print(f'Nested best-solo-model ({best_solo_key}) balanced accuracy: {np.mean(nested_scores_solo_best):.4f} (+/- {np.std(nested_scores_solo_best):.4f})')
print(f'Nested 3-way-blend balanced accuracy:                      {np.mean(nested_scores_blend):.4f} (+/- {np.std(nested_scores_blend):.4f})')
print(f'Honest improvement estimate: {np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best):+.4f}')

nested CV (3-way blend):   0%|          | 0/5 [00:00<?, ?it/s]

blend grid (xgb_ovr+catboost_v1+catboost_v2):   0%|          | 0/231 [00:00<?, ?it/s]

blend grid (xgb_ovr+catboost_v1+catboost_v2):   0%|          | 0/231 [00:00<?, ?it/s]

blend grid (xgb_ovr+catboost_v1+catboost_v2):   0%|          | 0/231 [00:00<?, ?it/s]

blend grid (xgb_ovr+catboost_v1+catboost_v2):   0%|          | 0/231 [00:00<?, ?it/s]

blend grid (xgb_ovr+catboost_v1+catboost_v2):   0%|          | 0/231 [00:00<?, ?it/s]

Per-fold blend weights (fit on the other 4 folds): [(0.4, 0.6, 0.0), (0.55, 0.2, 0.25), (0.4, 0.6, 0.0), (0.55, 0.2, 0.25), (0.5, 0.5, 0.0)]
Nested best-solo-model (xgb_ovr) balanced accuracy: 0.9493 (+/- 0.0014)
Nested 3-way-blend balanced accuracy:                      0.9494 (+/- 0.0012)
Honest improvement estimate: +0.0001


## Pairwise blends (for comparison)

Worth knowing whether a 2-way subset does as well as the full 3-way blend — e.g.
whether XGB-OvR pairs better with the same-feature-set CatBoost-V2 than with V1.

In [13]:
pair_grid = simplex_grid(2, step=0.02)
pairwise_results = {}
for pair in combinations(MEMBER_NAMES, 2):
    best_score, best_w = grid_search_blend(OOF_PROBAS, y, list(pair), pair_grid)
    pairwise_results[pair] = (best_score, best_w)

for pair, (score, w) in sorted(pairwise_results.items(), key=lambda kv: -kv[1][0]):
    print(f'{score:.4f}  {"+".join(pair)}  weights={tuple(round(x,2) for x in w)}')

print(f'\n3-way blend (same-data): {best_score_3way:.4f}')
print(f'Best solo model:         {max(solo_scores.values()):.4f}')

blend grid (xgb_ovr+catboost_v1):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (xgb_ovr+catboost_v2):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (catboost_v1+catboost_v2):   0%|          | 0/51 [00:00<?, ?it/s]

0.9495  xgb_ovr+catboost_v1  weights=(0.46, 0.54)
0.9494  xgb_ovr+catboost_v2  weights=(0.8, 0.2)
0.9493  catboost_v1+catboost_v2  weights=(1.0, 0.0)

3-way blend (same-data): 0.9495
Best solo model:         0.9493


## Decision + candidate submission

Use the nested-validation estimate to decide whether XGB-OvR (solo or blended)
beats the current best single model.

In [14]:
honest_improvement = np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best)
THRESHOLD_FOR_REAL_IMPROVEMENT = 0.0005
CURRENT_BEST_OOF = 0.9493

print(f'XGB-OvR solo:              {oof_bal_acc_ovr:.4f}')
print(f'CatBoost-V1 solo:          {result_cb_v1["oof_balanced_accuracy"]:.4f}')
print(f'CatBoost-V2 solo:          {result_cb_v2["oof_balanced_accuracy"]:.4f}')
print(f'Current project best OOF: {CURRENT_BEST_OOF:.4f}')
print()

if honest_improvement > THRESHOLD_FOR_REAL_IMPROVEMENT and (np.mean(nested_scores_blend) > CURRENT_BEST_OOF + THRESHOLD_FOR_REAL_IMPROVEMENT):
    print(f'REAL IMPROVEMENT: blend {honest_improvement:+.4f} over solo (nested-validated), and beats current best. Using weights fit on full OOF: {dict(zip(MEMBER_NAMES, best_w_3way))}')
    test_pred = blend_predict(TEST_PROBAS, best_w_3way, MEMBER_NAMES)
    submission = pd.DataFrame({'id': test['id'], TARGET: label_encoder.inverse_transform(test_pred)})
    assert list(submission.columns) == list(sample_submission.columns)
    assert len(submission) == len(sample_submission)
    assert set(submission[TARGET].unique()) <= set(label_encoder.classes_)
    assert submission[TARGET].isnull().sum() == 0
    submission_path = DATA_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'wrote {submission_path} ({len(submission)} rows) — NOT auto-submitted to Kaggle')
    display(submission[TARGET].value_counts(normalize=True))
elif oof_bal_acc_ovr > CURRENT_BEST_OOF + THRESHOLD_FOR_REAL_IMPROVEMENT:
    print(f'XGB-OvR solo beats the current best by {oof_bal_acc_ovr - CURRENT_BEST_OOF:+.4f} -- worth a submission on its own merits.')
    test_pred = test_proba_ovr.argmax(axis=1)
    submission = pd.DataFrame({'id': test['id'], TARGET: label_encoder.inverse_transform(test_pred)})
    assert list(submission.columns) == list(sample_submission.columns)
    assert len(submission) == len(sample_submission)
    assert set(submission[TARGET].unique()) <= set(label_encoder.classes_)
    assert submission[TARGET].isnull().sum() == 0
    submission_path = DATA_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'wrote {submission_path} ({len(submission)} rows) — NOT auto-submitted to Kaggle')
    display(submission[TARGET].value_counts(normalize=True))
else:
    print(f'NO REAL IMPROVEMENT over the current best (~{CURRENT_BEST_OOF:.4f} OOF). '
          f'Not writing a new submission.csv.')

XGB-OvR solo:              0.9493
CatBoost-V1 solo:          0.9493
CatBoost-V2 solo:          0.9491
Current project best OOF: 0.9493

NO REAL IMPROVEMENT over the current best (~0.9493 OOF). Not writing a new submission.csv.


## Summary

**A flat result by our own validation threshold, though the highest public
leaderboard score submitted in this project so far.**

All three solo scores land within noise of each other: XGB-OvR 0.9493 (an exact
tie with CatBoost-V1), CatBoost-V1 0.9493, CatBoost-V2 0.9491. The 3-way blend's
best same-data score (0.9495) and the nested-validated honest improvement
(+0.0001) are both far below the 0.0005 threshold we use to call an improvement
real, so we did not write a new candidate submission from the 3-way blend. The
best pairwise combination was `xgb_ovr + catboost_v1` (0.9495 at weights
0.46/0.54) — better than the full 3-way blend or either other pair, and notably
better than `catboost_v1 + catboost_v2` (0.9493, same as solo), confirming that
the two CatBoost variants remain too correlated with each other to help in a
blend.

We also submitted the best pairwise blend (`xgb_ovr + catboost_v1`) to check its
leaderboard score, even though it did not clear our validation threshold for a
confirmed improvement: **public LB 0.94937**, the highest score submitted in
this project so far (previous best: 0.94913). This is a genuine, if small, edge,
but it sits well inside the noise band our validation has repeatedly found on
this dataset (deltas across several rounds of tuning and ensembling are on the
order of ±0.0001 to ±0.0005). We treat this as statistically tied with our
existing best model rather than a confirmed new best.

A separate, independent analysis corroborates this conclusion. [Masaya
Kawamata's notebook](https://www.kaggle.com/code/masayakawamata/s6e7-xgb-ovr-cv-0-95036),
linked from the discussion above, runs a roughly 20-arm ablation campaign and
finds that the one-vs-rest decomposition itself is a no-op relative to native
multiclass training (+0.00003, correlation 0.99982), and that per-class
`scale_pos_weight` — the same imbalance handling used for XGB-OvR here — is
actively harmful when combined with a separate decision-rule correction. That
matches the double-correction pitfall documented in [Georgy Mamarin's
discussion thread](https://www.kaggle.com/competitions/playground-series-s6e7/discussion/717018)
on this same competition (a correction that helps in isolation becomes harmful
when stacked with a second, overlapping correction), making this a third
independent confirmation of that pattern.

Our existing CatBoost model remains the best model with a stable, credible
out-of-fold score behind it. Several independent attempts to squeeze further
gains out of this dataset — threshold tuning, ensembling, discussion-driven
research, and this XGBoost one-vs-rest test — now all point at the same
synthesis-noise ceiling, so further attempts here should be weighed against
diminishing, noise-level returns.
